# Topic 5: End-to-End AWS Pipeline Architecture

> **Phase 7 → Week 14 → Topic 5**

---

## The Full Picture

This notebook ties together everything from Week 14: Airflow, EMR, S3, Glue Catalog, monitoring — into a complete, production-grade reference architecture.

```
Sources → Bronze (S3 + Glue Catalog) → Silver (EMR/Glue) → Gold (EMR) → Athena/Redshift
             ↑                              ↑                  ↑              ↑
         Airflow DAG orchestrates all steps, monitors, alerts on failure
```

---

## Interview Cheat Sheet

**Q: Walk me through a production batch ETL pipeline on AWS end-to-end.**
> Source data (RDS, Kafka, vendor SFTP) lands in S3 Bronze via ingest jobs. An Airflow DAG runs daily: (1) S3KeySensor waits for all source files, (2) EMRCreateJobFlowOperator creates an ephemeral cluster with Spot Task nodes, (3) EmrAddStepsOperator submits Bronze → Silver → Gold Spark steps sequentially, (4) EmrStepSensors wait for each step, (5) data quality checks run after each layer, (6) EmrTerminateJobFlowOperator terminates the cluster (`trigger_rule=all_done`), (7) CloudWatch alarms fire if DQ checks fail or freshness SLA is missed.

**Q: How do you handle schema changes in a production pipeline without breaking downstream consumers?**
> Three strategies: (1) Schema registry (Glue Schema Registry / Confluent) — schemas are versioned, breaking changes require a new version, producers/consumers declare compatibility (BACKWARD, FORWARD, FULL). (2) Schema evolution with Delta Lake — `mergeSchema=true` adds new columns, old consumers see null for new fields (additive = safe). (3) Contract testing — downstream team tests against a fixture of the current schema; if producer changes schema, contract test fails in CI before deployment.

**Q: How do you make a pipeline idempotent?**
> Idempotent: re-running produces the same result, no duplicates. Techniques: (1) Write with `partitionOverwriteMode=dynamic` — only overwrite affected partitions, (2) Delta MERGE INTO on a unique key — upsert handles re-runs, (3) Use the execution date in the output path — each run writes to its own dated prefix, (4) For streaming: checkpoint ensures exactly-once processing, (5) For Glue bookmarks: reset and re-run only unprocessed files.

In [ ]:
print("End-to-End AWS Pipeline — Reference Architecture")
print("This notebook is a design guide; code patterns shown require AWS.")

## Part 1: Full Architecture Diagram

In [ ]:
print("""
Production AWS Data Pipeline — Full Architecture:
══════════════════════════════════════════════════════════════

INGEST LAYER
  ┌─────────────┐    ┌──────────────────────────────────────────┐
  │ Source RDS  │──▶ │ AWS DMS → S3 Bronze                      │
  │ Kafka topic │──▶ │ Kafka Connect → S3 Bronze (Parquet/Avro) │
  │ SFTP files  │──▶ │ Lambda → S3 Bronze                       │
  └─────────────┘    └─────────────────┬────────────────────────┘
                                        │  S3 event → SQS
                                        ▼
ORCHESTRATION
  ┌─────────────────────────────────────────────────────────────┐
  │ Apache Airflow (MWAA or self-managed on ECS)                │
  │                                                             │
  │  S3KeySensor ──▶ CreateEMRCluster ──▶ AddSteps             │
  │                       │                    │                │
  │  Bronze Step ──────────────────────────────┤                │
  │  Silver Step  ◀── EmrStepSensor            │                │
  │  DQ Check     ◀── PythonOperator           │                │
  │  Gold Step    ◀── EmrAddSteps              │                │
  │  Terminate    ◀── all_done trigger_rule    │                │
  └─────────────────────────────────────────────────────────────┘
                                        │
PROCESSING LAYER (EMR Cluster - ephemeral)
  ┌─────────────────────────────────────────────────────────────┐
  │ EMR 6.15 (Spark 3.5)                                        │
  │  MASTER: 1 × m5.xlarge (On-Demand)                         │
  │  CORE:   2 × m5.2xlarge (On-Demand)                        │
  │  TASK:   N × m5.2xlarge (Spot, 60-80% savings)             │
  │                                                             │
  │  Bronze job: read S3 raw → validate schema → write Bronze   │
  │  Silver job: deduplicate → cast types → enrich → DQ checks  │
  │  Gold job:   aggregate by date/region → write Gold          │
  └───────────────────┬─────────────────────────────────────────┘
                      │ reads/writes
STORAGE LAYER (S3 + Delta Lake)
  ┌─────────────────────────────────────────────────────────────┐
  │ s3://data-bucket/                                           │
  │   bronze/orders/year=2024/month=01/day=15/  (Parquet)      │
  │   silver/orders/date=2024-01-15/            (Delta Lake)   │
  │   gold/revenue/date=2024-01-15/             (Delta Lake)   │
  │   checkpoints/                               (Streaming)   │
  └──────────────────┬──────────────────────────────────────────┘
                     │ schema registered in
METADATA LAYER
  ┌─────────────────────────────────────────────────────────────┐
  │ Glue Data Catalog                                           │
  │   analytics_db.bronze_orders  → points to S3 Bronze        │
  │   analytics_db.silver_orders  → points to S3 Silver        │
  │   analytics_db.gold_revenue   → points to S3 Gold          │
  └──────────────────┬──────────────────────────────────────────┘
                     │ queried by
SERVING LAYER
  ┌─────────────────────────────────────────────────────────────┐
  │ Amazon Athena    → ad-hoc SQL on S3/Glue Catalog (no ETL)  │
  │ Amazon Redshift  → Redshift Spectrum (query S3 from RS)     │
  │ QuickSight       → BI dashboards on Athena/Redshift         │
  │ SageMaker        → ML training on Silver/Gold Delta tables  │
  └─────────────────────────────────────────────────────────────┘

OBSERVABILITY
  CloudWatch:  EMR cluster metrics, Airflow task metrics
  Airflow UI:  DAG run history, task logs, SLA tracking
  Spark UI:    Stage/task breakdown (via History Server on S3 logs)
  PagerDuty:   SNS → PagerDuty for P1 alerts
══════════════════════════════════════════════════════════════
""")

## Part 2: Complete Airflow DAG

In [ ]:
print("""
Complete Production Airflow DAG:
══════════════════════════════════════════════════════════════

from airflow import DAG
from airflow.providers.amazon.aws.operators.emr import (
    EmrCreateJobFlowOperator, EmrAddStepsOperator, EmrTerminateJobFlowOperator
)
from airflow.providers.amazon.aws.sensors.emr import EmrStepSensor
from airflow.providers.amazon.aws.sensors.s3 import S3KeySensor
from airflow.operators.python import PythonOperator
from airflow.utils.task_group import TaskGroup
from datetime import datetime, timedelta
import boto3, json

def run_dq_check(**context):
    layer = context['layer']
    date  = context['ds']
    # Read DQ metrics stored by the Spark job to S3
    s3 = boto3.client('s3')
    obj = s3.get_object(
        Bucket='my-bucket',
        Key=f'dq-metrics/{layer}/{date}/metrics.json'
    )
    metrics = json.loads(obj['Body'].read())
    null_rate = metrics['columns']['customer_id']['null_rate']
    if null_rate > 0.01:
        raise ValueError(f"{layer} DQ failed: customer_id null_rate={null_rate:.2%}")
    print(f"{layer} DQ passed")

CLUSTER_CONFIG = { ... }   # see notebook 02

BRONZE_STEP = [{'Name': 'Bronze', 'HadoopJarStep': {
    'Jar': 'command-runner.jar',
    'Args': ['spark-submit', '--deploy-mode', 'cluster',
             's3://bucket/scripts/bronze_ingest.py', '--date', '{{ ds }}']
}}]

SILVER_STEP = [{'Name': 'Silver', 'HadoopJarStep': {
    'Jar': 'command-runner.jar',
    'Args': ['spark-submit', 's3://bucket/scripts/silver_transform.py', '--date', '{{ ds }}']
}}]

GOLD_STEP = [{'Name': 'Gold', 'HadoopJarStep': {
    'Jar': 'command-runner.jar',
    'Args': ['spark-submit', 's3://bucket/scripts/gold_build.py', '--date', '{{ ds }}']
}}]

default_args = {
    'owner': 'data-eng', 'retries': 3,
    'retry_delay': timedelta(minutes=5),
    'email_on_failure': True, 'email': ['oncall@company.com']
}

with DAG('daily_etl', default_args=default_args,
         schedule_interval='0 6 * * *', start_date=datetime(2024,1,1),
         catchup=False, tags=['production', 'orders']) as dag:

    wait = S3KeySensor(
        task_id='wait_for_source',
        bucket_name='my-source-bucket',
        bucket_key='raw/{{ ds }}/_SUCCESS',
        mode='reschedule', timeout=7200, poke_interval=120,
    )

    create = EmrCreateJobFlowOperator(
        task_id='create_cluster', job_flow_overrides=CLUSTER_CONFIG
    )
    cluster_id = "{{ ti.xcom_pull('create_cluster', key='return_value') }}"

    with TaskGroup('bronze') as bronze_group:
        bronze_submit = EmrAddStepsOperator(
            task_id='submit', job_flow_id=cluster_id, steps=BRONZE_STEP
        )
        bronze_wait = EmrStepSensor(
            task_id='wait', job_flow_id=cluster_id,
            step_id="{{ ti.xcom_pull('bronze.submit')[0] }}",
            mode='reschedule', poke_interval=60
        )
        bronze_dq = PythonOperator(
            task_id='dq_check', python_callable=run_dq_check,
            op_kwargs={'layer': 'bronze'}, provide_context=True,
        )
        bronze_submit >> bronze_wait >> bronze_dq

    with TaskGroup('silver') as silver_group:
        # same pattern: submit → wait → dq_check
        pass

    with TaskGroup('gold') as gold_group:
        # same pattern
        pass

    terminate = EmrTerminateJobFlowOperator(
        task_id='terminate_cluster', job_flow_id=cluster_id,
        trigger_rule='all_done'   # ← always runs
    )

    wait >> create >> bronze_group >> silver_group >> gold_group >> terminate
══════════════════════════════════════════════════════════════
""")

## Part 3: Spark Job Template (Production-Ready)

In [ ]:
print("""
Production Spark Job Template (silver_transform.py):
══════════════════════════════════════════════════════════════

import argparse, json, sys, logging
from datetime import datetime
import boto3
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

logging.basicConfig(level=logging.INFO,
    format='%(asctime)s %(levelname)s %(name)s: %(message)s')
log = logging.getLogger('silver_transform')

def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument('--date',       required=True)  # 2024-01-15
    p.add_argument('--env',        default='prod')  # dev/staging/prod
    p.add_argument('--input-path', default='s3://my-bucket/bronze/orders/')
    p.add_argument('--output-path',default='s3://my-bucket/silver/orders/')
    p.add_argument('--dq-path',    default='s3://my-bucket/dq-metrics/silver/')
    return p.parse_args()

def build_spark(env: str) -> SparkSession:
    return SparkSession.builder \
        .appName(f"SilverTransform-{env}") \
        .config("spark.sql.adaptive.enabled", "true") \
        .config("spark.sql.shuffle.partitions", "200") \
        .config("spark.hadoop.fs.s3a.committer.name", "magic") \
        .getOrCreate()

def transform(spark, args):
    log.info(f"Reading bronze: {args.input_path}date={args.date}/")
    bronze = spark.read.parquet(f"{args.input_path}date={args.date}/")

    silver = (
        bronze
        .filter(F.col('customer_id').isNotNull() & (F.col('amount') > 0))
        .dropDuplicates(['order_id'])
        .withColumn('event_ts', F.to_timestamp('event_time'))
        .withColumn('amount', F.col('amount').cast('double'))
        .select('order_id', 'customer_id', 'amount', 'status', 'event_ts')
    )

    row_count = silver.count()
    log.info(f"Silver rows: {row_count}")

    # Write with dynamic partition overwrite (idempotent)
    spark.conf.set('spark.sql.sources.partitionOverwriteMode', 'dynamic')
    silver.withColumn('date', F.lit(args.date)) \
          .write.mode('overwrite').partitionBy('date') \
          .parquet(args.output_path)
    log.info(f"Written to {args.output_path}date={args.date}/")

    # Publish DQ metrics to S3 for Airflow DQ task
    metrics = compute_dq(silver, row_count)
    s3 = boto3.client('s3')
    s3.put_object(
        Bucket='my-bucket',
        Key=f'dq-metrics/silver/{args.date}/metrics.json',
        Body=json.dumps(metrics)
    )
    return row_count

def compute_dq(df, row_count):
    null_customer = df.filter(F.col('customer_id').isNull()).count()
    return {
        'total_rows': row_count,
        'columns': {'customer_id': {'null_rate': null_customer / row_count}}
    }

if __name__ == '__main__':
    args = parse_args()
    spark = build_spark(args.env)
    try:
        count = transform(spark, args)
        log.info(f"SUCCESS: {count} rows")
        sys.exit(0)
    except Exception as e:
        log.exception(f"FAILED: {e}")
        sys.exit(1)   # non-zero exit → EMR step FAILED → Airflow detects failure
    finally:
        spark.stop()
══════════════════════════════════════════════════════════════
""")

## Part 4: Schema Registry with Glue

In [ ]:
print("""
AWS Glue Schema Registry — Schema Versioning:
══════════════════════════════════════════════════════════════

Glue Schema Registry stores and versions Avro/JSON/Protobuf schemas.
Producers and consumers negotiate schema compatibility.

Create a registry and schema:
  glue = boto3.client('glue')

  glue.create_registry(RegistryName='orders-registry')

  glue.create_schema(
    RegistryId={'RegistryName': 'orders-registry'},
    SchemaName='order-event',
    DataFormat='AVRO',
    Compatibility='BACKWARD',   # new schema can read old data
    SchemaDefinition=json.dumps({
        'type': 'record',
        'name': 'OrderEvent',
        'fields': [
            {'name': 'order_id',     'type': 'string'},
            {'name': 'customer_id',  'type': 'string'},
            {'name': 'amount',       'type': 'double'},
            {'name': 'status',       'type': 'string'},
        ]
    })
  )

  # Register a new version (add optional 'region' field — backward compatible)
  glue.register_schema_version(
    SchemaId={'RegistryName': 'orders-registry', 'SchemaName': 'order-event'},
    SchemaDefinition=json.dumps({
        'type': 'record',
        'name': 'OrderEvent',
        'fields': [
            {'name': 'order_id',    'type': 'string'},
            {'name': 'customer_id', 'type': 'string'},
            {'name': 'amount',      'type': 'double'},
            {'name': 'status',      'type': 'string'},
            {'name': 'region',      'type': ['null', 'string'], 'default': None}  # optional
        ]
    })
  )

Compatibility modes:
  BACKWARD:   new schema can read data written with old schema (add optional fields)
  FORWARD:    old schema can read data written with new schema (add fields with defaults)
  FULL:       both — bidirectional (most restrictive)
  NONE:       no compatibility check (dangerous in production)

Use with Kafka (MSK):
  from aws_schema_registry import SchemaRegistryClient
  # Serializer embeds schema ID in every Kafka message
  # Consumer auto-fetches schema from registry → no hardcoded schemas
══════════════════════════════════════════════════════════════
""")

## Part 5: Cost Estimation for a Production Pipeline

In [ ]:
# Cost calculator for an EMR-based daily pipeline
def estimate_emr_cost(
    n_on_demand_nodes: int,
    n_spot_nodes: int,
    instance_type: str,
    on_demand_price: float,
    spot_discount: float,
    runtime_hours: float,
    days_per_month: int = 30
) -> dict:
    spot_price = on_demand_price * (1 - spot_discount)
    daily_on_demand = n_on_demand_nodes * on_demand_price * runtime_hours
    daily_spot      = n_spot_nodes      * spot_price      * runtime_hours
    daily_total     = daily_on_demand + daily_spot
    monthly_total   = daily_total * days_per_month

    return {
        "instance_type":      instance_type,
        "on_demand_nodes":    n_on_demand_nodes,
        "spot_nodes":         n_spot_nodes,
        "spot_discount":      f"{spot_discount:.0%}",
        "runtime_hours":      runtime_hours,
        "daily_cost_usd":     round(daily_total, 2),
        "monthly_cost_usd":   round(monthly_total, 2),
        "vs_all_on_demand":   round((n_on_demand_nodes + n_spot_nodes) * on_demand_price * runtime_hours * days_per_month, 2),
    }

# Example: 1 master + 2 core (On-Demand) + 8 task (Spot)
estimate = estimate_emr_cost(
    n_on_demand_nodes=3,     # master + 2 core
    n_spot_nodes=8,          # task nodes
    instance_type="m5.2xlarge",
    on_demand_price=0.384,   # $/hr for m5.2xlarge
    spot_discount=0.70,      # 70% cheaper on Spot
    runtime_hours=3.0,       # job takes 3 hours
    days_per_month=30
)

print("EMR Cost Estimate:")
for k, v in estimate.items():
    print(f"  {k:25s}: {v}")

savings_pct = (estimate['vs_all_on_demand'] - estimate['monthly_cost_usd']) / estimate['vs_all_on_demand'] * 100
print(f"\n  Savings vs all On-Demand: {savings_pct:.0f}%")
print(f"  All On-Demand monthly:    ${estimate['vs_all_on_demand']}")
print(f"  With Spot task nodes:     ${estimate['monthly_cost_usd']}")

## Exercises

1. Design the complete Airflow DAG for a pipeline that processes three source tables (orders, payments, users) in parallel for Bronze, then sequentially for Silver (orders must finish before payments, payments before a join step). Draw the task dependency graph.
2. Write a production-ready Spark script that accepts `--date`, `--env`, `--input-path`, `--output-path` arguments, reads Bronze, writes Silver, publishes DQ metrics to S3, and exits with code 1 on any exception.
3. Calculate the monthly AWS cost for your pipeline: 1 master m5.xlarge + 4 core m5.2xlarge (On-Demand) + 10 task m5.2xlarge (Spot at 70% discount), running 4 hours daily. Compare vs running an always-on EMR cluster 24/7.
4. What is the Glue Schema Registry and what compatibility mode would you use for an orders schema that has consumers who must be deployable independently of producers?
5. Your daily pipeline fails at the Silver step. The cluster was terminated by the `all_done` trigger. How do you re-run only the Silver → Gold steps without reprocessing Bronze? What changes to the pipeline would make partial re-runs easier?